# PharmaIntel-AI — from biomedical literature to safety intelligence

**CB4020 mini-project · case study: clozapine and blood-cell adverse-event candidates**

This notebook is a cold-start, reproducible walkthrough. It loads the pinned, precomputed project outputs shipped under `dashboard/data/`, so it runs without downloading model weights or calling external APIs. If the local raw PubMed corpus is available, the optional inference command can reproduce the model run. Outputs below are rendered snapshots; rerun cells to refresh them.

> This is a literature-mining research prototype. It does not establish clinical causality, incidence, or treatment advice. Model scores and the exploratory cross-source index are not probabilities.


## Part 1 — Environment and tools

The pipeline uses Python, pandas, NetworkX, Hugging Face Transformers, and PyTorch. NCBI E-utilities is used only when retrieving PubMed records; the pinned Hugging Face model runs locally. The hosted dashboard reads precomputed files for its research results and makes no model-inference API calls; its live literature carousel separately fetches recent PubMed records through a shared six-hour cache.


In [1]:
import gzip, json, math, re, sys
from pathlib import Path
import networkx as nx
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the PharmaIntel repository checkout.")
ASSETS = ROOT / "dashboard/data"
LOCAL = ROOT / "data/processed/pubmed"
if (LOCAL / "transformer_case_study_evidence.csv").exists():
    evidence_path = LOCAL / "transformer_case_study_evidence.csv"
    summary_path = ROOT / "results/transformer_corpus_summary.json"
    signal_path = ROOT / "results/transformer_case_study_signals.csv"
    graph_path = ROOT / "results/pharmaintel_transformer_case_graph.graphml"
else:
    evidence_path = ASSETS / "transformer_case_study_evidence.csv.gz"
    summary_path = ASSETS / "transformer_corpus_summary.json"
    signal_path = ASSETS / "transformer_case_study_signals.csv"
    graph_path = ASSETS / "pharmaintel_transformer_case_graph.graphml.gz"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
evidence = pd.read_csv(evidence_path, dtype={"pmid": str}, keep_default_na=False)
signals = pd.read_csv(signal_path, keep_default_na=False)
print("Project root: <repository checkout>")
print("Evidence rows:", f"{len(evidence):,}", "| unique PMIDs:", f"{evidence.pmid.nunique():,}")
print("Model:", summary["model"]["model_id"], "| revision:", summary["model"]["model_revision"][:12])


Project root: <repository checkout>
Evidence rows: 1,148 | unique PMIDs: 314
Model: Glasgow-AI4BioMed/bioner_bc5cdr | revision: 6eaba08f3ae3


## Part 2 — PubMed data acquisition and provenance

The study corpus is retrieved through NCBI PubMed E-utilities using `Bio.Entrez`. Source records retain PMID, title, journal, publication date, query, and retrieval metadata. To retrieve/refresh the research corpus locally, configure an identifying email and run:

```bash
export NCBI_EMAIL="your-email@example.org"
python -m pharmaintel.data_acquisition.pubmed --max-results 200
```

`NCBI_API_KEY` is optional. It increases request limits; it is not an LLM API key. The hosted app uses the pinned snapshot for research results. Its literature carousel checks NCBI E-utilities with a server-side shared cache refreshed every six hours; it sends no user-entered text. A result is a literature mention, not a validated ADR alert.


In [2]:
print("Corpus snapshot:")
print(f"  {summary['sentences_scanned']:,} sentences across {summary['unique_pmids_scanned']:,} PubMed records")
print("  Evidence fields:", ", ".join(evidence.columns[:12]))
print("\nExample provenance rows (IDs and titles only):")
print(evidence[["pmid", "title", "publication_date"]].drop_duplicates("pmid").head(4).to_string(index=False))


Corpus snapshot:
  3,915 sentences across 385 PubMed records
  Evidence fields: pmid, record_type, title, journal, publication_date, sentence_index, drug, drug_surface, adr, adr_surface, drug_model_label, adr_model_label

Example provenance rows (IDs and titles only):
    pmid                                             title publication_date
26266027                   Olanzapine-Induced Neutropenia.      2015 Feb 24
32815018                Clozapine-induced agranulocytosis.         2020 Nov
42624624 Zuclopenthixol and clozapine-induced neutropenia.      2026 Aug 20
37910212 [Clozapine-induced Tubulointerstitial Nephritis].      2023 Aug 31


## Part 3 — Text preprocessing

PubMed abstracts are split into sentences while retaining the original sentence text and source offsets/provenance fields. We create a separate normalized view for analysis; we do not overwrite the source text used for evidence display. The current sentence splitter is a deterministic baseline, not a clinically validated language processor.


In [3]:
def normalize_for_matching(text: str) -> str:
    """Normalize whitespace and case for candidate matching; preserve source separately."""
    return re.sub(r"\s+", " ", str(text)).strip().casefold()

evidence["normalized_sentence"] = evidence["evidence_sentence"].map(normalize_for_matching)
print("Normalized evidence sentences:", f"{evidence.normalized_sentence.nunique():,}")
print("Original sentence column retained:", "evidence_sentence" in evidence.columns)
print("Normalization is case/whitespace only; no unsupported lemmatization is claimed.")


Normalized evidence sentences: 710
Original sentence column retained: True
Normalization is case/whitespace only; no unsupported lemmatization is claimed.


## Part 4 — Biomedical NER and drug–ADR relations

The pinned `Glasgow-AI4BioMed/bioner_bc5cdr` token classifier emits broad **Chemical** and **Disease** labels. PharmaIntel preserves these raw labels; a Chemical mention is not necessarily a drug, and a Disease mention is not necessarily an ADR. A pair-scoped lexical baseline labels sentence cues as supported, speculative, negated, or co-mention. These labels are review cues, not causal conclusions.


In [4]:
print("Raw NER labels:", summary["entity_counts_by_raw_label"])
print("Relation-cue status counts:", summary["case_study_relation_status_counts"])
print("\nTraceable example schema:")
print("  PMID → sentence → entity labels/scores → relation/context cues")
print("  Scores are transformer token-classification outputs, not clinical probabilities.")


Raw NER labels: {'Chemical': 3286, 'Disease': 4112}
Relation-cue status counts: {'co_mention': 861, 'negated': 10, 'speculative': 53, 'supported': 224}

Traceable example schema:
  PMID → sentence → entity labels/scores → relation/context cues
  Scores are transformer token-classification outputs, not clinical probabilities.


## Part 5 — Signal prioritization

The implemented literature score is `log(1 + positive unique PMIDs) × assertion consistency`. It deduplicates sentence evidence to the paper level and downweights speculative/mixed assertion context.

For the exact clozapine–neutropenia pair, Assignment 1 reports 929 FAERS reports and Assignment 2 reports EBGM 50.81. The dashboard adds an **exploratory, uncalibrated cross-source index** using fixed transforms and weights. It is not machine-learned, validated confidence, a probability, an incidence estimate, or a clinical risk estimate. It is displayed for this exact pair only; missing FAERS values for other terms are not imputed.


In [5]:
print("Top literature-ranked terms:")
print(signals[["drug", "adr", "unique_pmid_count", "supported_pmid_count", "consistency", "priority_score"]].to_string(index=False))
row = signals.loc[signals["adr"].str.casefold().eq("neutropenia")].iloc[0]
faers_reports, ebgm = 929, 50.81
components = {
    "FAERS EBGM": ebgm / (1 + ebgm),
    "Literature breadth": 1 - math.exp(-float(row.unique_pmid_count) / 50),
    "Supported relation share": float(row.supported_pmid_count) / max(float(row.unique_pmid_count), 1),
    "Assertion consistency": float(row.consistency),
}
weights = {"FAERS EBGM": .30, "Literature breadth": .25, "Supported relation share": .25, "Assertion consistency": .20}
index = 100 * sum(components[k] * weights[k] for k in components)
print(f"\nAssignment inputs: {faers_reports} FAERS reports; EBGM {ebgm:.2f}")
print("Exploratory cross-source index:", round(index, 1), "/ 100 (not calibrated)")
print("Components:", {k: round(v, 3) for k,v in components.items()})


Top literature-ranked terms:
     drug              adr  unique_pmid_count  supported_pmid_count  consistency  priority_score
clozapine      neutropenia                119                    93     0.869748        4.126895
clozapine  agranulocytosis                 65                    56     0.930769        3.899602
clozapine       leukopenia                  6                     6     1.000000        1.945910
clozapine granulocytopenia                  4                     4     1.000000        1.609438
clozapine      neutropenic                  1                     0     0.500000        0.346574

Assignment inputs: 929 FAERS reports; EBGM 50.81
Exploratory cross-source index: 89.0 / 100 (not calibrated)
Components: {'FAERS EBGM': 0.981, 'Literature breadth': 0.907, 'Supported relation share': 0.782, 'Assertion consistency': 0.87}


## Part 6 — Visualizations and evidence graph

Saved figures are generated reproducibly from the pinned dashboard assets by `python scripts/build_research_figures.py`. They use the PharmaIntel palette and show a term ranking and a source-traceable network.

![Literature ranking](../results/figures/literature_signal_ranking.svg)

![Clozapine–neutropenia evidence network](../results/figures/clozapine_neutropenia_evidence_graph.svg)


In [6]:
if graph_path.suffix == ".gz":
    with gzip.open(graph_path, "rt", encoding="utf-8") as handle:
        graph = nx.read_graphml(handle)
else:
    graph = nx.read_graphml(graph_path)
print(f"Evidence graph: {graph.number_of_nodes():,} nodes, {graph.number_of_edges():,} edges")
for image_name in ["literature_signal_ranking.svg", "clozapine_neutropenia_evidence_graph.svg"]:
    image_path = ROOT / "results/figures" / image_name
    print(image_name, "—", "available" if image_path.exists() else "run scripts/build_research_figures.py")


Evidence graph: 1,469 nodes, 3,444 edges
literature_signal_ranking.svg — available
clozapine_neutropenia_evidence_graph.svg — available


## Part 7 — Reporting, evaluation, and limitations

NER benchmark metrics are mention-level exact-span metrics on the manually annotated sample and must remain separated by sampling stratum. Relation-level precision/recall/F1 are **not available**: the 60-record PMID-disjoint blind sample still requires independent human labels. The notebook intentionally does not invent them.

1. Complete `dashboard/relation_annotation_app.py` with a blinded reviewer.
2. Run `python scripts/evaluate_relation_holdout.py` after all labels are complete.
3. Report held-out metrics only if the annotations are independent of rule development.
4. Review errors, publication bias, term mapping, and the limits of co-mention and lexical scope.

Final interpretation: this pipeline helps prioritize literature for human review. It does not replace formal signal detection, pharmacovigilance assessment, or medical judgment. The PDF report is a separate submission artifact.


In [7]:
print("Evaluation state:")
print(summary["evaluation"])
print("\nRows and signals in packaged snapshot:", f"{len(evidence):,}", "and", f"{len(signals):,}")
print("Held-out relation metrics: pending blinded human annotation")
print("\nOptional local reproduction:")
print("  python scripts/run_transformer_corpus_ner.py")
print("  python scripts/build_research_figures.py")


Evaluation state:
No corpus-level precision/recall/F1 reported because this corpus has no independent sentence-level NER or relation gold labels.

Rows and signals in packaged snapshot: 1,148 and 5
Held-out relation metrics: pending blinded human annotation

Optional local reproduction:
  python scripts/run_transformer_corpus_ner.py
  python scripts/build_research_figures.py


## SocialPulse — free public-news feed

The dashboard retrieves recent clozapine coverage and blood-cell safety news from GDELT, a public news index, with a six-hour shared cache. The query covers up to three months and requests at most 250 article records; coverage is not exhaustive, and users follow publisher links for full text. The official X API is pay-per-use, so PharmaIntel does not query X or scrape it. SocialPulse keeps news separate from PubMed and FAERS scores. The optional authorized, de-identified CSV/text sample screen is keyword-based only and is not a validated social ADR model.
